# Lecture 4 — Class Exercise
## Scatter & Bubble Charts: Gapminder

> **Push to:** `week04/lecture04_exercise.ipynb`

**Rules:**
1. Colour used **sparingly** — one categorical variable, no rainbow
2. If showing all continents, either use accessible palette OR grey all + highlight one
3. `size_max` set when using bubble size
4. Log scale for GDP per capita
5. Insight title

---


In [1]:
# Connect Google Drive (Colab only)
from google.colab import drive
drive.mount('/content/drive')

# Folder in your Drive where outputs (HTML charts) will be saved
import os
SAVE_DIR = '/content/drive/MyDrive/week04'
os.makedirs(SAVE_DIR, exist_ok=True)
print('Saving to:', SAVE_DIR)

Mounted at /content/drive
Saving to: /content/drive/MyDrive/week04


In [6]:
import math
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go


# Dataset: Gapminder — GDP, Life Expectancy, Population by Country
# Source: Gapminder Foundation (gapminder.org)

df = px.data.gapminder()
print(f"Loaded: {len(df)} rows")
print(df.head())

Loaded: 1704 rows
       country continent  year  lifeExp       pop   gdpPercap iso_alpha  \
0  Afghanistan      Asia  1952   28.801   8425333  779.445314       AFG   
1  Afghanistan      Asia  1957   30.332   9240934  820.853030       AFG   
2  Afghanistan      Asia  1962   31.997  10267083  853.100710       AFG   
3  Afghanistan      Asia  1967   34.020  11537966  836.197138       AFG   
4  Afghanistan      Asia  1972   36.088  13079460  739.981106       AFG   

   iso_num  
0        4  
1        4  
2        4  
3        4  
4        4  


In [7]:
# explore

print(df.info())
print("Years:", sorted(df['year'].unique()))
print("Continents:", df['continent'].unique())
print(df.describe().round(1))

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1704 entries, 0 to 1703
Data columns (total 8 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   country    1704 non-null   object 
 1   continent  1704 non-null   object 
 2   year       1704 non-null   int64  
 3   lifeExp    1704 non-null   float64
 4   pop        1704 non-null   int64  
 5   gdpPercap  1704 non-null   float64
 6   iso_alpha  1704 non-null   object 
 7   iso_num    1704 non-null   int64  
dtypes: float64(2), int64(3), object(3)
memory usage: 106.6+ KB
None
Years: [np.int64(1952), np.int64(1957), np.int64(1962), np.int64(1967), np.int64(1972), np.int64(1977), np.int64(1982), np.int64(1987), np.int64(1992), np.int64(1997), np.int64(2002), np.int64(2007)]
Continents: ['Asia' 'Europe' 'Africa' 'Americas' 'Oceania']
         year  lifeExp           pop  gdpPercap  iso_num
count  1704.0   1704.0  1.704000e+03     1704.0   1704.0
mean   1979.5     59.5  2.960121e+07     7215.3    

## Task 1 — Scatter: life expectancy change over time

**What to build:** A scatter showing **GDP per capita vs life expectancy** for **two years** (2002 and 2007) to show how both moved — use **colour for year** (just 2 colours), **one continent only**.

Choose any continent except Africa (that was the example). Highlight the change direction.

> 💡 Filter: `df.loc[df['continent'] == 'YOUR_CHOICE']` then filter years


In [8]:
# Task 1
# YOUR CODE HERE
# Task 1
# Task 1
continent = 'Asia'
d1 = df.loc[(df['continent'] == continent) & (df['year'].isin([2002, 2007]))].copy()
d1['year'] = d1['year'].astype(str)   # treat year as a category -> exactly 2 colours

fig1 = px.scatter(
    d1, x='gdpPercap', y='lifeExp',
    color='year',
    color_discrete_map={'2002': 'lightgrey', '2007': '#1f77b4'},  # grey = before, blue = after
    category_orders={'year': ['2002', '2007']},
    hover_name='country',
    log_x=True,
    template='plotly_white',
    labels={'gdpPercap': 'GDP per Capita (USD, log scale)',
            'lifeExp': 'Life Expectancy (years)', 'year': 'Year'},
    title='Between 2002 and 2007, nearly every Asian country moved up and to the right: richer and longer-lived'
)
fig1.update_traces(marker=dict(size=9, opacity=0.85))

# Arrows from each country's 2002 point to its 2007 point = change direction
p = d1.pivot(index='country', columns='year', values=['gdpPercap', 'lifeExp'])
for country, r in p.iterrows():
    fig1.add_annotation(
        x=math.log10(r[('gdpPercap', '2007')]), y=r[('lifeExp', '2007')],
        ax=math.log10(r[('gdpPercap', '2002')]), ay=r[('lifeExp', '2002')],
        xref='x', yref='y', axref='x', ayref='y',
        showarrow=True, arrowhead=2, arrowsize=1, arrowwidth=1,
        arrowcolor='rgba(31,119,180,0.5)', text=''
    )

# Label a few notable points directly
for c in ['China', 'Iraq', 'Kuwait']:
    r = d1[(d1['country'] == c) & (d1['year'] == '2007')].iloc[0]
    fig1.add_annotation(x=math.log10(r['gdpPercap']), y=r['lifeExp'], text=c,
                        showarrow=True, arrowhead=0, ax=25, ay=-25,
                        font=dict(size=12, family='Arial'))

fig1.update_layout(font=dict(family='Arial', size=13), height=550,
                   legend=dict(orientation='h', x=0, y=1.08, title=''))
fig1.show()
fig1.write_html(f'{SAVE_DIR}/task1_asia_scatter.html')

## Task 2 — Bubble chart: tell a story

**What to build:** A bubble chart (full 2007 dataset, all countries) where:
- x = GDP per capita (log scale)
- y = life expectancy
- size = population
- colour = ONE continent highlighted (your choice), all others grey
- At least one annotation explaining the highlighted group's story

> This is the grey-and-highlight technique applied to a bubble chart.


In [17]:
# Task 2
df07 = df.loc[df['year'] == 2007].copy()

highlight = 'Asia'
colour_map = {c: ('crimson' if c == highlight else 'lightgrey') for c in df07['continent'].unique()}

fig2 = px.scatter(
    df07, x='gdpPercap', y='lifeExp',
    size='pop', size_max=60,                 # cap bubble size
    color='continent', color_discrete_map=colour_map,
    hover_name='country',
    log_x=True,
    opacity=0.75,
    template='plotly_white',
    labels={'gdpPercap': 'GDP per Capita (USD, log scale)',
            'lifeExp': 'Life Expectancy (years)', 'pop': 'Population', 'continent': ''},
    title='Asia holds the world\'s two largest populations, yet they sit mid-table on wealth and health'
)

# China and India labelled directly
for country, ax, ay in [('China', 50, -45), ('India', 40, 50)]:
    r = df07[df07['country'] == country].iloc[0]
    fig2.add_annotation(x=math.log10(r['gdpPercap']), y=r['lifeExp'], text=country,
                        showarrow=True, arrowhead=1, ax=ax, ay=ay,
                        arrowcolor='crimson',
                        font=dict(size=13, family='Arial', color='crimson'))

# Story annotation
fig2.add_annotation(
    x=math.log10(40000), y=50,
    text='<b>Asia (red)</b><br>China and India alone hold<br>over 2 billion people, yet remain<br>well below the wealthy West',
    showarrow=False, align='left',
    font=dict(color='crimson', size=12, family='Arial'),
    bgcolor='white', bordercolor='crimson', borderwidth=1, borderpad=4
)

fig2.update_layout(font=dict(family='Arial', size=13), showlegend=False, height=600)
fig2.show()
fig2.write_html(f'{SAVE_DIR}/task2_asia_bubble.html')